# Ticket Transformer trained from scratch

This notebook uses the same installed commands as the CLI. Open it from a clone of this repository after the README setup. The Jupyter frontend can run in another environment; training always uses the root uv environment. Default: tiny smoke, not the full benchmark.


In [ ]:
import json
import subprocess
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "uv.lock").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Open the notebook from a clone of this repository")
    ROOT = ROOT.parent


def run(*args):
    return subprocess.run(
        ["uv", "run", "--no-sync", "pretraining", *map(str, args)], cwd=ROOT, check=True
    )


print(
    subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
)
print("Environment:", ROOT / ".venv")

## Data and training

Set FULL_RUN=True only for the documented full base model. The smoke verifies the workflow; its scores are not the published benchmark. Run IDs preserve earlier outputs.


In [ ]:
FULL_RUN = False
CONFIG = (
    ROOT / "pretraining/configs" / ("ticket-base.toml" if FULL_RUN else "smoke.toml")
)
RUN_ID = "notebook-" + datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
run("data", "prepare", "--config", CONFIG)
run("train", "--config", CONFIG, "--run-id", RUN_ID)
ARTIFACT = ROOT / "pretraining/runs" / RUN_ID
print(json.loads((ARTIFACT / "run_info.json").read_text()))

## Actual inference

The model generates constrained intent and priority tokens; Python maps the labels to JSON. Untrained/smoke outputs can be wrong.


In [ ]:
run(
    "predict",
    "--model",
    ARTIFACT,
    "--text",
    "My invoice is incorrect. There is no rush; next week is fine.",
)
run("evaluate", "--config", CONFIG, "--model", ARTIFACT, "--split", "val")

## Read the frozen benchmark

The following plots are committed results from the completed three-seed experiment, not outputs from the smoke above. See the report for all individual seeds and uncertainty. Classical baselines remain stronger on this synthetic benchmark.


In [ ]:
from IPython.display import Image, display

REPORT = ROOT / "pretraining/reports/ticket-v2"
print((REPORT / "results.md").read_text())
display(Image(filename=str(REPORT / "comparison.png")))
display(Image(filename=str(REPORT / "learning-curves.png")))

## Repeat the complete benchmark

Run this separately from interactive model selection:

```bash
uv run --no-sync pretraining benchmark --config pretraining/configs/ticket-pool.toml --output pretraining/runs/notebook-benchmark
```

The matrix selects on validation before final scoring. Launch the demo with the saved selected artifact using `pretraining demo --model ARTIFACT --report pretraining/reports/ticket-v2`. Dataset and model cards explain the synthetic-data limitations. CUDA/Colab execution has not been validated by this CPU run.
